In [ ]:
from __future__ import print_function
import argparse
from math import log10
from datetime import datetime

import tensorflow as tf

from model import ConvNet
from dataset import DatasetFromFolder

parser = argparse.ArgumentParser(description='TensorFlow Super Res Example')

import sys
import os
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from model import ConvNet
#from data import get_dataset
from dataset import DatasetFromFolder
#from SelfDefLoss import PhysicsLoss

#os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
#print(torch.cuda.device_count())
#print(torch.cuda.is_available())
#device = torch.device("cuda") #cuda or cpu
#torch.backends.cudnn.benchmark = True
device = torch.device("cpu") #cuda or cpu

parser = argparse.ArgumentParser(description='PyTorch Super Res Example')
parser.add_argument('--seed', nargs='?', const=500, type=int, default=500, help='seed value to use')
parser.add_argument('--epochs', nargs='?', const=1000, type=int, default=1000, help='number of epochs')
parser.add_argument('--lr', nargs='?', const=0.001, type=float, default=0.001, help='learning rate')
parser.add_argument('--restart', nargs='?', const=0, type=int, default=0, help='restart switch and location')
opt = parser.parse_args()

tf.random.set_seed(opt.seed)
nEpochs = opt.epochs
lr = opt.lr
restart = opt.restart

print('===> Loading datasets')
training_data_loader = DatasetFromFolder(
    '/gpfs/scratch/zexizhang/Sediment/data/train/input/',
    '/gpfs/scratch/zexizhang/Sediment/data/train/target/',
    batch_size=256,
    shuffle=True,
    transform=False,
)
test_data_loader = DatasetFromFolder(
    '/gpfs/scratch/zexizhang/Sediment/data/test/input/',
    '/gpfs/scratch/zexizhang/Sediment/data/test/target/',
    batch_size=256,
    shuffle=False,
)

print('===> Building model')
model = ConvNet()
model.build((None, None, None, 8))

try:
    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=0.01)
except AttributeError:
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

loss_fn = tf.keras.losses.MeanSquaredError()

if restart:
    model.load_weights("./trained_model/model_epoch_{}.weights.h5".format(restart))


def train(epoch):
    epoch_loss = 0.0
    start = datetime.now()
    for iteration, (inputs, targets) in enumerate(training_data_loader, 1):
        with tf.GradientTape() as tape:
            predictions = model(inputs, training=True)
            loss = loss_fn(targets, predictions)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        epoch_loss += float(loss.numpy())
        print("===> Epoch[{}]({}/{}): Loss: {:.8f}".format(epoch, iteration, len(training_data_loader), float(loss.numpy())))
        print("Training MSE {}".format(float(loss.numpy())))

    if epoch % 1000 == 0:
        new_lr = float(optimizer.learning_rate.numpy()) * 0.7
        optimizer.learning_rate.assign(new_lr)

    print("===> Epoch {} Complete: Avg. Loss: {:.8f}".format(epoch, epoch_loss / len(training_data_loader)))
    with open('./trained_model/TrainingError.dat', 'a') as f:
        print("{:.8f}".format(epoch_loss / len(training_data_loader)), file=f)
    print("Epoch " + str(epoch) + " completed in: " + str(datetime.now() - start))


def test():
    avg_psnr = 0
    for inputs, targets in test_data_loader:
        predictions = model(inputs, training=False)
        loss = loss_fn(targets, predictions)
        mse = float(loss.numpy())
        print("Test MSE {}".format(mse))
        psnr = 10 * log10(1 / mse)
        avg_psnr += psnr
    print("===> Avg. PSNR: {:.4f} dB".format(avg_psnr / len(test_data_loader)))
    with open('./trained_model/TestingError.dat', 'a') as f:
        print("{:.8f}".format(avg_psnr / len(test_data_loader)), file=f)


def checkpoint(epoch):
    model_out_path = "./trained_model/model_epoch_{}.weights.h5".format(epoch)
    model.save_weights(model_out_path)
    print("Checkpoint saved to {}".format(model_out_path))


if restart:
    start = restart + 1
else:
    start = 1
    file = open('./trained_model/TrainingError.dat', 'w')
    file.close()
    file = open('./trained_model/TestingError.dat', 'w')
    file.close()


for epoch in range(start, nEpochs + start):
    train(epoch)
    if epoch % 100 == 0:
        checkpoint(epoch)
    test()
